# Summit-Sim: Complete E2E Workflow Demo

Demonstrates the full Summit-Sim workflow:
1. **Scenario Generation** - AI generates wilderness rescue scenario
2. **Simulation** - Student navigates scenario with human-in-the-loop
3. **Debrief** - Performance analysis and learning report

All phases are traced to MLflow for visibility.

## 1. Setup & Imports

In [1]:
import warnings

# Suppress autologging warnings
warnings.filterwarnings("ignore", message=".*autologging.*")
warnings.filterwarnings("ignore", message=".*LangChain.*")

from summit_sim.agents.generator import generate_scenario
from summit_sim.graphs.simulation import create_simulation_graph
from summit_sim.schemas import HostConfig, generate_scenario_id
from summit_sim.settings import settings
from summit_sim.tracing import enable_tracing, simulation_session

# Initialize MLflow tracing
enable_tracing()

print(f"✓ MLflow: {settings.mlflow_tracking_uri}")
print(f"✓ Experiment: {settings.mlflow_experiment_name}")
print(f"✓ Model: {settings.default_model}")
print(f"✓ API Key: {bool(settings.openrouter_api_key)}")

✓ MLflow: https://mlflow.bhamm-lab.com
✓ Experiment: summit-sim
✓ Model: google/gemini-3.1-flash-lite-preview
✓ API Key: True


## 2. Phase 1: Scenario Generation

Generate a complete wilderness rescue scenario from minimal host configuration.

In [2]:
# Configuration
config = HostConfig(
    num_participants=3,
    activity_type="hiking",
    difficulty="med",
    class_id="demo-class-2024",
)

print(
    f"Generating: {config.activity_type} scenario for {config.num_participants} participants"
)
print("-" * 60)

# Generate scenario
scenario = await generate_scenario(config)
scenario_id = generate_scenario_id()

print(f"✓ Title: {scenario.title}")
print(f"✓ Setting: {scenario.setting}")
print(f"✓ Patient: {scenario.patient_summary}")
print(f"✓ Turns: {len(scenario.turns)}")
print(f"✓ Scenario ID: {scenario_id}")
print("\nLearning Objectives:")
for obj in scenario.learning_objectives:
    print(f"  • {obj}")

Generating: hiking scenario for 3 participants
------------------------------------------------------------
✓ Title: The Ridge Line Collapse
✓ Setting: High-alpine trail in the North Cascades, approximately 10,500 feet elevation. Weather is cold, windy, with rapidly rolling fog. The group is 4 hours from the trailhead.
✓ Patient: 35-year-old female, fit, experienced hiker, no known medical history. Complaining of extreme fatigue, headache, and a non-productive cough while at 10,500ft in the North Cascades.
✓ Turns: 4
✓ Scenario ID: scn-fcdd3f8e

Learning Objectives:
  • Perform a rapid focused assessment in a high-altitude wilderness setting.
  • Recognize the difference between mild Acute Mountain Sickness (AMS) and potentially fatal High Altitude Pulmonary Edema (HAPE).
  • Prioritize immediate descent as the primary wilderness treatment for HAPE.


Trace(trace_id=tr-76c25dfb91253bd1a3ae11093441adc7)

### Display Scenario Structure

In [3]:
for turn in scenario.turns:
    marker = "(START)" if turn.is_starting_turn else ""
    print(f"\n{'=' * 60}")
    print(f"Turn {turn.turn_id} {marker}")
    print(f"{'=' * 60}")
    print(
        turn.narrative_text[:200] + "..."
        if len(turn.narrative_text) > 200
        else turn.narrative_text
    )
    print("\nChoices:")
    for choice in turn.choices:
        mark = "✓" if choice.is_correct else " "
        next_turn = (
            "END" if choice.next_turn_id is None else f"Turn {choice.next_turn_id}"
        )
        print(f"  [{mark}] {choice.description[:60]}... → {next_turn}")


Turn 0 (START)
Your group of three is hiking along an exposed ridge at 10,500 feet. Sarah, one of your hikers, sits on a rock, clutching her chest. She is clearly struggling to catch her breath, has a loud, dry coug...

Choices:
  [ ] Assume she is just deconditioned and push to keep the group ... → Turn 1
  [✓] Stop immediately, perform a focused primary assessment (ABCD... → Turn 2

Turn 1 
By choosing to keep pushing, Sarah’s condition has deteriorated. She is now cyanotic (bluish tint) around her lips and her cough sounds wet, not dry. She is barely able to stand. You realize you MUST ...

Choices:
  [ ] Rest for 30 minutes to see if she recovers, then re-evaluate... → Turn 2
  [✓] Recognize the emergency and start descent immediately.... → Turn 2

Turn 2 
Your assessment reveals a respiratory rate of 30, tachycardia, and audible crackles in her lungs. You are now certain this is HAPE. You need to decide on evacuation strategy immediately before the wea...

Choices:
  [ ] Administ

## 3. Phase 2: Simulation

Run the interactive simulation with human-in-the-loop.
The graph will pause at each turn for you to make a choice.

In [4]:
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command

# Create graph with memory
memory = InMemorySaver()
graph = create_simulation_graph(checkpointer=memory)

# Initialize state
initial_state = {
    "scenario_draft": scenario,
    "current_turn_id": scenario.starting_turn_id,
    "transcript": [],
    "is_complete": False,
    "key_learning_moments": [],
    "last_selected_choice": None,
    "simulation_result": None,
    "scenario_id": scenario_id,
    "class_id": config.class_id,
    "debrief_report": None,
}

print("✓ Graph initialized")
print(f"✓ Starting at turn {initial_state['current_turn_id']}")

✓ Graph initialized
✓ Starting at turn 0


In [5]:
# Run simulation with MLflow session tracking
from summit_sim.tracing import log_simulation_results

with simulation_session(config, scenario_id=scenario_id) as (session_id, graph_config):
    print(f"\n📋 Session ID: {session_id}")
    print("🎮 Starting simulation...\n")
    print("=" * 60)
    print("INTERACTIVE SIMULATION")
    print("=" * 60 + "\n")

    current_state = initial_state
    turn_count = 0
    simulation_complete = False
    first_turn = True

    while not simulation_complete:
        turn_count += 1

        # Get the current turn from the graph state
        current_turn_id = current_state["current_turn_id"]
        current_turn = scenario.get_turn(current_turn_id)

        if current_turn is None:
            print(f"Error: Turn {current_turn_id} not found")
            break

        # Display turn
        print(f"\n--- TURN {turn_count} ---\n")
        print(current_turn.narrative_text)
        print("\nChoices:")
        for i, choice in enumerate(current_turn.choices, 1):
            print(f"  {i}. {choice.description}")

        # Get user input
        while True:
            try:
                sel = int(input("\nSelect choice (number): ")) - 1
                if 0 <= sel < len(current_turn.choices):
                    break
                print("Invalid selection")
            except ValueError:
                print("Please enter a number")

        selected_choice = current_turn.choices[sel]

        # Process the turn using ainvoke to maintain tracing context
        if first_turn:
            # First turn: pass full state to initialize the graph
            current_state = await graph.ainvoke(
                initial_state,
                graph_config,
            )
            first_turn = False
        else:
            # Subsequent turns: use Command to resume
            current_state = await graph.ainvoke(
                Command(resume={"choice_id": selected_choice.choice_id}),
                graph_config,
            )

        # Check if complete
        if current_state.get("is_complete"):
            simulation_complete = True

        # Safety limit
        if turn_count > 10:
            print("\nSafety limit reached")
            break

    print("\n" + "=" * 60)
    print("SIMULATION COMPLETE")
    print("=" * 60)

    # Log results
    log_simulation_results(
        transcript=current_state["transcript"],
        is_complete=current_state["is_complete"],
        key_learning_moments=current_state["key_learning_moments"],
        debrief_report=current_state["debrief_report"],
    )
    print("\n✓ Results logged to MLflow")
    print(f"✓ Total turns: {len(current_state['transcript'])}")
    print(f"✓ Learning moments: {len(current_state['key_learning_moments'])}")


📋 Session ID: 51dff429-7180-42f9-ad42-7947415f6d35
🎮 Starting simulation...

INTERACTIVE SIMULATION


--- TURN 1 ---

Your group of three is hiking along an exposed ridge at 10,500 feet. Sarah, one of your hikers, sits on a rock, clutching her chest. She is clearly struggling to catch her breath, has a loud, dry cough, and admits she feels 'heavy' and confused. The weather is turning, with fog rapidly obscuring the trail.

Choices:
  1. Assume she is just deconditioned and push to keep the group moving to reach the campsite faster.
  2. Stop immediately, perform a focused primary assessment (ABCDE) and check vitals.


2026/03/24 06:58:18 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during autologging: <Token var=<ContextVar name='current_context' default={} at 0x7f7d8facf560> at 0x7f7d5316d480> was created in a different Context
2026/03/24 06:58:18 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during autologging: <Token var=<ContextVar name='current_context' default={} at 0x7f7d8facf560> at 0x7f7d531ac500> was created in a different Context
2026/03/24 06:58:18 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during autologging: <Token var=<ContextVar name='current_context' default={} at 0x7f7d8facf560> at 0x7f7da9a07480> was created in a different Context



--- TURN 2 ---

Your group of three is hiking along an exposed ridge at 10,500 feet. Sarah, one of your hikers, sits on a rock, clutching her chest. She is clearly struggling to catch her breath, has a loud, dry cough, and admits she feels 'heavy' and confused. The weather is turning, with fog rapidly obscuring the trail.

Choices:
  1. Assume she is just deconditioned and push to keep the group moving to reach the campsite faster.
  2. Stop immediately, perform a focused primary assessment (ABCDE) and check vitals.


2026/03/24 06:58:20 WARNING mlflow.tracing.fluent: Failed to start span LangGraph: 'NoneType' object has no attribute 'set_span_type'. For full traceback, set logging level to debug.
Deserializing unregistered type summit_sim.schemas.ScenarioDraft from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('summit_sim.schemas', 'ScenarioDraft')]
2026/03/24 06:58:20 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during autologging: <Token var=<ContextVar name='current_context' default={} at 0x7f7d8facf560> at 0x7f7d531f7300> was created in a different Context
2026/03/24 06:58:22 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during autologging: <Token var=<ContextVar name='current_context' default={} at 0x7f7d8facf560> at 0x7f7d5316c980> was created in a different Context
2026/03/24 06:58:22 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during autologging: <Token var=<ContextVa


--- TURN 3 ---

Your assessment reveals a respiratory rate of 30, tachycardia, and audible crackles in her lungs. You are now certain this is HAPE. You need to decide on evacuation strategy immediately before the weather closes in completely.

Choices:
  1. Administer ibuprofen and try to move her 100 feet down to a sheltered spot and camp for the night.
  2. Initiate an immediate, unassisted, rapid descent of at least 2,000 feet, keeping her from exerting herself.


Deserializing unregistered type summit_sim.schemas.ScenarioDraft from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('summit_sim.schemas', 'ScenarioDraft')]
Deserializing unregistered type summit_sim.schemas.ChoiceOption from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('summit_sim.schemas', 'ChoiceOption')]
Deserializing unregistered type summit_sim.schemas.SimulationResult from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('summit_sim.schemas', 'SimulationResult')]
2026/03/24 06:58:30 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during autologging: <Token var=<ContextVar name='current_context' default={} at 0x7f7d8facf560> at 0x7f7d5320f440> was created in a different Context
2026/03/24 06:58:35 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during autologging: <Token var=<ContextVa


--- TURN 4 ---

By prioritizing rapid descent, you successfully brought Sarah to a lower elevation. Her breathing improved significantly within an hour of descending below 8,000 feet, and she was able to walk the remaining distance to the trailhead with assistance. Had you stayed at 10,000 feet, her condition would likely have resulted in respiratory failure. Scenario Complete.

Choices:
  1. Accept the outcome.
  2. Try again.


2026/03/24 06:58:37 WARNING mlflow.tracing.fluent: Failed to start span LangGraph: 'NoneType' object has no attribute 'set_span_type'. For full traceback, set logging level to debug.
Deserializing unregistered type summit_sim.schemas.ScenarioDraft from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('summit_sim.schemas', 'ScenarioDraft')]
Deserializing unregistered type summit_sim.schemas.ChoiceOption from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('summit_sim.schemas', 'ChoiceOption')]
Deserializing unregistered type summit_sim.schemas.SimulationResult from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('summit_sim.schemas', 'SimulationResult')]
2026/03/24 06:58:37 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during autologging: <Token var=<ContextVar name='current_context' default={} at 0x7f7d8facf560> a


SIMULATION COMPLETE

✓ Results logged to MLflow
✓ Total turns: 3
✓ Learning moments: 6
🏃 View run sim-demo-class-2024-hiking-3p-med at: https://mlflow.bhamm-lab.com/#/experiments/7/runs/9c559102b6704f35b1e2539233aa767e
🧪 View experiment at: https://mlflow.bhamm-lab.com/#/experiments/7


[Trace(trace_id=tr-1cd87cfd89464fe70968b50d528065e6), Trace(trace_id=tr-6aa71b04b6af4b7ca09d10f7635628dd), Trace(trace_id=tr-216960d5dc73aa5680c570a4486ca4f9), Trace(trace_id=tr-8dcce2b8df2670248df790311617ec33), Trace(trace_id=tr-0105b849e676e65b9177107d2d236bfd)]

## 4. Phase 3: Debrief

View the comprehensive performance report generated automatically by the simulation graph.

In [6]:
debrief_report = current_state["debrief_report"]

print("=" * 60)
print("DEBRIEF REPORT")
print("=" * 60)

print(f"\n📊 Final Score: {debrief_report.final_score:.1f}%")
status_emoji = "✅" if debrief_report.completion_status == "pass" else "❌"
print(f"{status_emoji} Status: {debrief_report.completion_status.upper()}")

print("\n📝 Summary:")
print(debrief_report.summary)

if debrief_report.key_mistakes:
    print("\n⚠️ Key Mistakes:")
    for mistake in debrief_report.key_mistakes:
        print(f"  • {mistake}")

if debrief_report.strong_actions:
    print("\n💪 Strong Actions:")
    for action in debrief_report.strong_actions:
        print(f"  • {action}")

if debrief_report.teaching_points:
    print("\n📚 Teaching Points:")
    for point in debrief_report.teaching_points:
        print(f"  • {point}")

if debrief_report.best_next_actions:
    print("\n🎯 Recommendations:")
    for rec in debrief_report.best_next_actions:
        print(f"  • {rec}")

DEBRIEF REPORT

📊 Final Score: 66.7%
❌ Status: FAIL

📝 Summary:
The student demonstrated strong initial assessment skills but struggled with the clinical priority of treating HAPE. While the rapid identification of a potential medical issue was correct, the subsequent decision to 'wait out' the condition by camping instead of initiating an immediate descent presented a significant risk to the patient. This highlights an area for improvement regarding the urgency and definitive treatment protocols required for altitude-related emergencies.

⚠️ Key Mistakes:
  • Chose to camp for the night rather than initiating an immediate descent, which is the only definitive treatment for HAPE in a wilderness setting.

💪 Strong Actions:
  • Successfully prioritized a systematic (ABCDE) assessment during the initial turn, which is crucial for establishing a baseline for respiratory distress.

📚 Teaching Points:
  • HAPE is a progressive and potentially fatal condition; any symptom of it necessitates a

## 5. Results Summary

Complete transcript and learning moments from the simulation.

In [7]:
print("\n" + "=" * 60)
print("COMPLETE TRANSCRIPT")
print("=" * 60 + "\n")

for i, entry in enumerate(current_state["transcript"], 1):
    status = "✓" if entry["was_correct"] else "✗"
    print(f"Turn {i} {status}")
    print(f"  Situation: {entry['turn_narrative'][:80]}...")
    print(f"  Choice: {entry['choice_description']}")
    print(f"  Feedback: {entry['feedback'][:100]}...")
    print()


COMPLETE TRANSCRIPT

Turn 1 ✓
  Situation: Your group of three is hiking along an exposed ridge at 10,500 feet. Sarah, one ...
  Choice: Stop immediately, perform a focused primary assessment (ABCDE) and check vitals.
  Feedback: Excellent decision. Prioritizing a systematic assessment is critical when a healthy hiker suddenly s...

Turn 2 ✗
  Situation: Your assessment reveals a respiratory rate of 30, tachycardia, and audible crack...
  Choice: Administer ibuprofen and try to move her 100 feet down to a sheltered spot and camp for the night.
  Feedback: Staying put with HAPE is unfortunately dangerous, as this condition can progress rapidly to fatal le...

Turn 3 ✓
  Situation: By prioritizing rapid descent, you successfully brought Sarah to a lower elevati...
  Choice: Accept the outcome.
  Feedback: Excellent work. By prioritizing rapid descent at the first signs of respiratory distress, you preven...



In [8]:
print("\n" + "=" * 60)
print("KEY LEARNING MOMENTS")
print("=" * 60 + "\n")

for i, moment in enumerate(current_state["key_learning_moments"], 1):
    print(f"{i}. {moment}")


KEY LEARNING MOMENTS

1. Systematic assessments (ABCDE) prevent cognitive bias and ensure you don't miss life-threatening pathologies.
2. In a wilderness setting, sudden respiratory distress at high altitude must be treated as HAPE until proven otherwise.
3. HAPE is a life-threatening medical emergency where immediate descent is the only definitive treatment.
4. Audible crackles (rales) upon auscultation indicate pulmonary fluid accumulation, which rapidly worsens at altitude and cannot be managed by rest or NSAIDs alone.
5. HAPE can escalate rapidly; recognizing the early sign of a dry cough at high altitude allows for actionable intervention before the patient's condition becomes critical.
6. Descent is the definitive treatment for HAPE; even a drop of a few thousand feet can significantly improve oxygenation and stabilize the patient.


---
## ✅ Complete Workflow Demo

All phases executed:
- ✓ Scenario generated with AI
- ✓ Simulation completed with human-in-the-loop
- ✓ Debrief report generated
- ✓ All traces logged to MLflow

View traces in MLflow UI at your configured tracking URI.